# Do the edits that work agree on a direction? — latent edit-direction alignment across four architectures

**Thread:** `notebooks/experiments/editability/latent_linearity/` · **branch** `latent_linearity` · sub-question 3
(editability) with a bearing on sub-question 1 (state geometry). **No models were trained for this notebook.**

## What this asks

This repo has found four edit mechanisms that reliably move a world model's *generation* to the edited world.
Each of them therefore hands us a **ground-truth latent displacement** — what a successful edit *is*, as a vector:

$$\Delta h \;=\; h(\text{model told about the edit}) \;-\; h(\text{matched control that was not})$$

`delta_h_analysis` (2026-08-03) measured, on the GRU and the RSSM, that the two oracle mechanisms agree on that
vector (cos +0.80 / +0.57). This notebook asks whether that agreement is **a property of latent world state or a
property of those two checkpoints**, by extending it to four architectures, and whether the two *learned* edit
pathways — an action channel the model was trained on, and a single post-edit observation on a model that has seen
teleports — write the **same** displacement as the oracles that need no training at all.

Two things this notebook is **not** about: it does not evaluate editors (no probe-derived writes appear anywhere),
and it does not touch compositionality or superposition. It is about the geometry of the displacements themselves.

## The four mechanisms

| # | mechanism | what the model is given | applies to |
|---|---|---|---|
| 1 | **Counterfactual Overwriting** | a whole rewritten history in which the object always travelled to the target, teacher-forced from scratch | every model |
| 2 | **Freeze-time Interp. TF @8** | 8 extra rendered frames interpolating the object to the target with the world otherwise frozen, appended to the real history | every model |
| 3 | **Action Interface** | the same observations, with the teleport issued through the **action channel the model was trained on** | only a model trained with teleport actions |
| 4 | **First Obs. TF** | one post-edit observation, teacher-forced onto the real history | every model, but it only *persists* on a model that saw teleports in training |

Mechanisms 1 and 2 always apply, so they carry **Part 1** (all four architectures). Mechanisms 3 and 4 need a
particular training distribution, so they carry **Part 2** — which is GRU-only, and §5 says why.

## Provenance

Part 1: dataset `datasets/4_fixed_refl_inview`, `edits` split, N=256 held-out episodes, edit frame 20, 15-step
rollouts. Part 2: `datasets/15_teleport_eval_single/eval.h5`, a **teleport-free** world with the single edit
synthesised, N=256, via `scripts/eval_action_sweep.xg_data`. Mechanisms and geometry come from
`edit_directions.py`; the §4 scorecard from `scripts/editability_metrics.py`; figures from `figures.py`; run
definitions from `LATENT_LINEARITY_RUNS.md` beside this notebook.


## Definitions — read this before any number

### The models, and which state object is analysed

An architecture can carry more than one thing that deserves the name "state", and they come apart
(`findings/architecture-independence.md`, 2026-08-04), so the unit of analysis is the **(checkpoint, state
object)** pair and every figure names it. Full configurations: `LATENT_LINEARITY_RUNS.md`.

| label used in every figure | run code | registry | state analysed | `H` |
|---|---|---|---|---|
| GRU · hidden state | `H256` | `../controls/CONTROL_RUNS.md` | the recurrent hidden state `h` | 256 |
| RSSM · det+stoch state | `4_dset4_refined_best` | `../../../../research/scratch/2026-06-29-rssm-refinement.md` | `cat(h_det, s_stoch)`, prior mean, `sample=False` | 320 |
| Transformer · residual stream | `W16` | `../transformers/TRANSFORMER_RUNS.md` | residual stream at the final point (4), current position | 256 |
| Latent DiT · latent window | `0_latent_dit_z16_w4` | `../latent_DiT/LATENT_DIT_RUNS.md` | the carried 4×16 latent buffer | 64 |
| DiT (pixel) · residual stream | `9_dset4_dit_w4_d256` | `../DiT/DIT_RUNS.md` | final-block token features, current position | 256 |

*Why these state objects.* The GRU and RSSM have exactly one. The transformer and the pixel DiT carry a buffer of
raw **observations** and recompute a residual stream; the buffer is not a learned representation at all, so the
residual stream is the analogue of `h` and is what is analysed. The latent DiT is the opposite case: its carried
state *is* a learned code, so the latent window is analysed rather than its (denoising-contaminated) activations.
Both diffusion variants are shown because they differ in exactly this respect.

### Conventions that every number below depends on

| convention | what it means | why |
|---|---|---|
| **predictive state** | the state whose `decode` gives frame `t+1`. Identity for the GRU / transformer / DiT family; one prior (`imagine`) step for the RSSM, whose decoder reconstructs the current frame | comparing states at different world-times is the one error a Δh study cannot absorb; §0 measures the alignment rather than asserting it |
| **matched control** | every Δh is `edited − control`, where the control is the *same construction with the edit removed* — the true history re-rendered, the frames frozen at the pre-edit position, the no-op action, the unedited frame | it removes re-rendering, the extra frames, and the noise draw, leaving only the edit |
| **same noise draw** | a mechanism's edited and control frames get **identical** observation noise (σ = 0.2, the training value) | independent draws inject a difference unrelated to the edit that is large enough to dominate the cosines |
| **alignment of the arms** | mechanisms 1–3 leave the model about to predict frame `ef`; mechanism 4 has consumed frame `ef` and is about to predict `ef+1`. Cross-mechanism comparisons free-run the first three one step so all four sit at `ef+1`; the 1-vs-2 pair is also reported at `ef` | `First Obs. TF` genuinely leads by one frame — the canonical rule is to label it, never to re-align the others to it |

### Metrics — formula, units, better-direction

Registry rows: `../METRICS_AND_EDITORS.md` §4 (the scorecard) and §5 (the Δh geometry, added with this notebook).

| name | formula | units | better |
|---|---|---|---|
| **Edit Index** | `(d_uned − d_edit)/(d_uned + d_edit)`, `d_·` = RMSE to each ground-truth world over the rays where they differ; per sample then averaged | −1 … +1 | ↑ (+1 = the edited world) |
| **Edit Index by step** | the same at every rollout step, against the counterfactual world rolled forward | −1 … +1 | ↑ |
| **GT-traj RMSE** | mean over the 15-step rollout of RMSE to the sim's **clean** post-edit render | obs intensity | ↓ |
| **fidelity ratio** | GT-traj RMSE(arm) ÷ GT-traj RMSE(its own unsteered) | ratio | ↓; **> 1 = the edit degraded the rollout** |
| **latent edit-direction cosine** | `cos(Δh_a, Δh_b)` per episode, then averaged; chance = 0 at every `H` | — | ↑ |
| **shuffled-pair control** | the same cosine with `Δh_b` taken from a *different* episode | — | the measured chance level |
| **projection fraction** | `mean |cos|` — the share of one displacement's magnitude lying along the other | 0 … 1 | ↑ |
| **enrichment** | projection fraction ÷ its shuffled control | × chance | ↑; **this is the cross-architecture number**, because `mean |cos|` for unrelated vectors falls as `√(2/πH)` and `H` runs 64 … 320 here |
| **‖Δh‖ ÷ ‖h‖** | edit size against the unedited state's own norm | fraction | — (a scale, not a score) |
| **‖Δh‖ ÷ one dynamics step** | edit size against `‖h_t − h_{t−1}‖` on the pre-edit context | ratio | — |
| **direction consistency** | mean cosine between the Δh of **different** episodes; chance 0 (`1/√H` is the per-pair sd, not a floor) | — | ↑ = a shared "an object moved" axis exists |
| **probe row-space fraction** | `‖P_row·Δh‖/‖Δh‖` with `P_row = A⁺A` from the standard linear position probe, reported **÷ chance `√(d/H)`** | × chance | — (1.0 = as visible as a random direction) |

Position probes are the standard `pim.extractors.fit_readability_probes` (linear least squares plus a 2×256 MLP,
both fit on the same 80% of *sequences* and scored on the held-out 20%), fit on **predictive** states so the probe
is applied to the same kind of state it was fit on. `d = 4` (two objects × (x, y)).


In [ ]:
# [1] Setup — the model roster, the edits split, and the shared constants.
import os, sys, time
from pathlib import Path

# `scripts/eval_action_sweep.py` (imported in Part 2) resolves `runs/` and `datasets/` relative to the
# process working directory, exactly as it does when run from the command line. So the notebook adopts
# the repo root as its working directory and every path below is repo-relative — rather than each call
# site carrying its own `../../../..`, which is where a stale path hides.
THREAD = Path.cwd()
REPO = THREAD.parents[3]
sys.path.insert(0, str(THREAD))
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "scripts"))
os.chdir(REPO)

import h5py
import numpy as np
import torch
from IPython.display import Markdown, display

import edit_directions as ed
import figures as F
from editability_metrics import (
    build_edit_zones,
    edit_scorecard,
    fidelity_ratio,
    random_samples,
    shift_zones,
)
from pim.figures import waterfall_grid
from pim.world_models import load_dataset

torch.manual_seed(0)
np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

N_EVAL, K_ROLL, N_OBJ, N_FT, N_PROBE, N_CTX = 256, 15, 2, 8, 400, 6
FIGDIR = "runs/latent_linearity/figures"
os.makedirs(FIGDIR, exist_ok=True)

UNS = "Unsteered (no edit)"
CF, FT, AC, FO = ed.COUNTERFACTUAL, ed.FREEZE_TIME, ed.ACTION, ed.FIRST_OBS

SPECS = [
    ed.ModelSpec("gru", "GRU · hidden state (H=256)", "runs/controls/H256/best_model.pt",
                 "GRU", "hidden state h", run_code="H256", registry="../controls/CONTROL_RUNS.md"),
    ed.ModelSpec("rssm", "RSSM · det+stoch state (H=320)", "runs/rssm/4_dset4_refined_best/best_model.pt",
                 "RSSM", "det+stoch state", run_code="4_dset4_refined_best"),
    ed.ModelSpec("trf", "Transformer · residual stream (H=256)", "runs/transformers/W16/best_model.pt",
                 "Transformer", "residual stream, final point", state_view="activations", probe_layer=4,
                 run_code="W16", registry="../transformers/TRANSFORMER_RUNS.md"),
    ed.ModelSpec("ldit", "Latent DiT · latent window (H=64)", "runs/latent_dit/0_latent_dit_z16_w4/best_model.pt",
                 "Latent DiT", "carried latent window", state_view="latent_window",
                 run_code="0_latent_dit_z16_w4", registry="../latent_DiT/LATENT_DIT_RUNS.md"),
    ed.ModelSpec("dit", "DiT (pixel) · residual stream (H=256)", "runs/dit/9_dset4_dit_w4_d256/best_model.pt",
                 "DiT (pixel)", "residual stream, final block", state_view="activations",
                 run_code="9_dset4_dit_w4_d256", registry="../DiT/DIT_RUNS.md"),
]
MODELS = [s.label for s in SPECS]

bundle = load_dataset("datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ)
edits, test = bundle.edits, bundle.test
ef, sim, R = edits.edit_frame, test.config["dataset"]["sim"], edits.obs_res
with h5py.File(edits.h5_path, "r") as f:
    VEL = f["velocities"][:, :, :N_OBJ, :].astype(np.float32)

idx = np.arange(N_EVAL)
edit_obj = edits.edit_object[idx].astype(int)
obs = edits.obs[idx].astype(np.float32)
gt_roll = edits.clean_obs[idx, ef : ef + K_ROLL].astype(np.float32)

print(f"working directory {Path.cwd()}")
print(f"device {DEVICE} · dataset 4 edits split · N={N_EVAL} episodes · edit frame {ef} · "
      f"rollout {K_ROLL} steps · obs_res {R}")
print(f"observation noise sigma={sim['obs_noise_std']} · position noise sigma={sim['position_noise_std']}")


In [ ]:
# [2] Build every mechanism's evidence (rendered once, shared by all five models) and the ray zones.
t0 = time.perf_counter()
EV = ed.build_evidence(
    positions=edits.positions[idx][:, :, :N_OBJ, :],
    velocities=VEL[idx],
    edit_object=edit_obj,
    target=edits.positions[idx, ef][np.arange(N_EVAL), edit_obj],
    sim=sim,
    ef=ef,
    n_obj=N_OBJ,
    n_ft=N_FT,
    seed=0,
)
ZONES = build_edit_zones(
    pre_pos=EV.pre_pos,
    tgt_pos=EV.tgt_pos,
    pre_vel=VEL[idx, ef - 1],
    edit_object=EV.edit_object,
    sim=sim,
    n_obj=N_OBJ,
    traj_pos=edits.positions[idx, ef : ef + K_ROLL, :N_OBJ, :].astype(np.float32),
    gt_edited_traj=gt_roll,
)
ZONES1 = shift_zones(ZONES, 1, gt_roll)  # the alignment `First Obs. TF` is scored at

print(f"rendered {N_EVAL} counterfactual histories + freeze-time runs + post-edit frames "
      f"in {time.perf_counter() - t0:.0f}s")
print(f"teleport {ZONES.teleport.mean():.2f} sim units (min {ZONES.teleport.min():.2f}, "
      f"max {ZONES.teleport.max():.2f})")
print(f"rays per episode — target {ZONES.target.sum(1).mean():.1f} · ghost {ZONES.ghost.sum(1).mean():.1f} "
      f"· differing {ZONES.differing.sum(1).mean():.1f} of {R} (the Edit Index scores only these)")


In [ ]:
# [3] One pass per model: alignment check, the three mechanisms' states, rollouts, scorecards, probe, deltas.
probe_obs = test.obs[:N_PROBE].astype(np.float32)
probe_pos = test.positions[:N_PROBE, :, :N_OBJ, :].reshape(N_PROBE, -1, N_OBJ * 2)
obs_test = torch.from_numpy(test.obs[:256].astype(np.float32)).to(DEVICE)
ARMS = [UNS, CF, FT, FO]

RES = {}
for spec in SPECS:
    t0 = time.perf_counter()
    model = ed.load_model(spec, root=".", device=DEVICE)

    align = ed.alignment_profile(model, obs_test, test.clean_obs[:256])
    run = ed.build_states(model, spec, EV, obs, ef, device=DEVICE)

    rolls = {UNS: ed.rollout(model, run.unsteered, K_ROLL)}
    for arm in (CF, FT, FO):
        rolls[arm] = ed.rollout(model, run.mechanisms[arm].edited, K_ROLL)

    # `First Obs. TF` leads by one frame, so it is scored against the shifted worlds AND against an
    # unsteered reference advanced by the same step — never against the edit-frame one.
    uns_lead = edit_scorecard(
        ed.rollout(model, ed.advance(model, run.unsteered, 1), K_ROLL - 1), ZONES1, gt_roll[:, 1:]
    )
    cards = {}
    for arm, roll in rolls.items():
        lead = arm == FO
        z, g = (ZONES1, gt_roll[:, 1:]) if lead else (ZONES, gt_roll)
        cards[arm] = edit_scorecard(roll[:, : g.shape[1]], z, g)
        cards[arm]["fidelity_ratio"] = fidelity_ratio(cards[arm], uns_lead if lead else cards[UNS])

    RES[spec.label] = dict(
        spec=spec,
        align=align,
        cards=cards,
        rolls=rolls,
        probe=ed.fit_position_probe(model, probe_obs, probe_pos, device=DEVICE, seed=0),
        dh_ef=ed.deltas(model, run, align_to_edit_frame=True),
        dh_next=ed.deltas(model, run, align_to_edit_frame=False),
        h0_norm=run.h0_norm,
        step_norm=run.step_norm,
    )
    print(f"{spec.label:44s} {time.perf_counter() - t0:5.0f}s")
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


---
## §0 — Is every model's state aligned to the same frame?

A Δh comparison is only meaningful if each model's analysed state is *about the same moment*. This is measured,
not asserted: on ordinary (non-edit) test sequences, `decode(predictive(state))` is scored against
`clean_obs[t+1+k]` for `k ∈ {−2, −1, 0, +1}`, averaged over `t = 12 … 30`.

Read the profile against two references printed beside it. **`frame change`** is the RMSE between consecutive
clean frames — the distance a correctly-aligned prediction has to cover. A predictor that hedges toward the frame
it just saw can score *lower* at `k = −1` than at `k = 0` while still being a correctly-aligned next-frame
predictor; that is a statement about how far it moves, not about which frame it is on. The decisive check is
whether the `k = 0` value reproduces the model's **independently published next-step RMSE against the clean
render**, which is quoted in the table from each model's own registry.


In [ ]:
# [4] Table 1 — the alignment check, against each model's own published next-step RMSE vs the clean render.
# Published references, copied from the registries named in the definitions table (cite, do not recompute):
#   GRU H256          0.1041   `../controls/CONTROL_RUNS.md` (noise-ablation "both" cell, 2026-07-30)
#   RSSM refined      0.1067   `research/scratch/2026-06-29-rssm-refinement.md` (near-horizon MSE 0.01726 -> 0.1314
#                              all-horizon; the near-frame value re-derived here, so quoted as "n/a" rather than
#                              a number on a different horizon convention)
#   Transformer W16   n/a      the registry reports val MSE vs the NOISY target only
#   Latent DiT z16 W4 0.1083   `../latent_DiT/LATENT_DIT_RUNS.md` val MSE vs clean 0.01174
#   DiT concat W4     0.1089   `../latent_DiT/LATENT_DIT_RUNS.md` cross-thread row, val MSE vs clean 0.01186
PUBLISHED = {
    "GRU · hidden state (H=256)": "0.1041",
    "RSSM · det+stoch state (H=320)": "n/a",
    "Transformer · residual stream (H=256)": "n/a",
    "Latent DiT · latent window (H=64)": "0.1083",
    "DiT (pixel) · residual stream (H=256)": "0.1089",
}
rows = ["| model | k=−2 | k=−1 | **k=0** | k=+1 | argmin | frame change | published next-step RMSE vs clean |",
        "|---|---|---|---|---|---|---|---|"]
for m in MODELS:
    a = RES[m]["align"]
    p = a["profile"]
    rows.append(
        f"| {m} | {p[-2]:.4f} | {p[-1]:.4f} | **{p[0]:.4f}** | {p[1]:.4f} | k={a['argmin']:+d} | "
        f"{a['frame_change']:.4f} | {PUBLISHED[m]} |"
    )
display(Markdown(
    "**Table 1 — frame alignment of the analysed state.** RMSE of `decode(predictive(state))` against "
    "`clean_obs[t+1+k]`, averaged over t = 12 … 30 on 256 ordinary test sequences. `frame change` is "
    "RMSE(clean_obs[t+1], clean_obs[t]) over the same window.\n\n" + "\n".join(rows)
))

for m in MODELS:
    pr = RES[m]["probe"]
    print(f"{m:44s} position readout R²  linear {pr['linear_r2']:.3f}  MLP {pr['mlp_r2']:.3f}")


---
## §1 — The gate: do these mechanisms actually edit the generation?

Δh is only worth analysing if the states it is built from produce the edited world. Every mechanism is scored on
the canonical §4 set at the edit frame **and across the whole rollout** — landing an edit and holding it are
different results — and the qualitative panel is the arbiter, because a scorecard cannot distinguish "the edit
worked" from "the output degraded".

`First Obs. TF` is included here even though it belongs to Part 2's question, because it is the reference every
architecture *can* be given and its weakness on models that never saw a teleport is the contrast Part 2 turns
into a result.


In [ ]:
# [5] Table 2 + Fig 1 — the canonical scorecard, and the Edit Index for every model and mechanism.
rows = ["| model | mechanism | Edit Index (step 0) ↑ | Edit Index (step 14) ↑ | GT-traj RMSE ↓ | "
        "Target RMSE ↓ | Ghost RMSE ↓ | Collateral RMSE ↓ | fidelity ↓ |", "|---|---|---|---|---|---|---|---|---|"]
for m in MODELS:
    for arm in ARMS:
        c = RES[m]["cards"][arm]
        zone = "n/a | n/a | n/a" if arm == FO else (
            f"{c['target_rmse']:.3f} | {c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f}")
        rows.append(
            f"| {m} | {arm} | **{c['edit_index']:+.3f}** | {c['edit_index_by_step'][-1]:+.3f} | "
            f"{c['gt_traj_rmse']:.3f} | {zone} | {c['fidelity_ratio']:.2f} |"
        )
display(Markdown(
    "**Table 2 — do the mechanisms edit the generation?** N=256 held-out episodes. `First Obs. TF` is scored one "
    "frame later than the others (it consumed the edit frame) and against an unsteered reference advanced by the "
    "same step; its zone RMSEs read `n/a` because the target / ghost / collateral ray masks are defined by the "
    "render at the edit frame and are not carried forward.\n\n" + "\n".join(rows)
))

fig = F.plot_edit_gate(
    {a: [RES[m]["cards"][a]["edit_index"] for m in MODELS] for a in ARMS},
    {a: [RES[m]["cards"][a]["fidelity_ratio"] for m in MODELS] for a in ARMS},
    MODELS, ARMS,
    title=f"Fig 1 — does each mechanism move the generation to the edited world?  N={N_EVAL} held-out edits",
)
fig.savefig(f"{FIGDIR}/fig1_gate.png", dpi=140, bbox_inches="tight")


In [ ]:
# [6] Fig 2 — Edit Index across the rollout: landing the edit and holding it are different results.
fig = F.plot_index_by_step(
    {m: {a: RES[m]["cards"][a]["edit_index_by_step"] for a in ARMS} for m in MODELS},
    MODELS, ARMS,
    title="Fig 2 — Edit Index at every rollout step, per architecture",
)
fig.savefig(f"{FIGDIR}/fig2_by_step.png", dpi=140, bbox_inches="tight")


In [ ]:
# [7] Fig 3a-e — the canonical waterfall, one per architecture. Samples are RANDOM (seed 0), not selected.
def ray_centre(mask: np.ndarray) -> np.ndarray:
    """Mean ray index of a boolean zone mask, per episode — the locator's x position."""
    return np.array([np.where(r)[0].mean() if r.any() else np.nan for r in mask])


SAMPLES = random_samples(N_EVAL, 4, seed=0)
TGT_X, GHOST_X = ray_centre(ZONES.target), ray_centre(ZONES.ghost)
print(f"random episodes shown in every waterfall: {SAMPLES}  "
      f"(their teleports {np.round(ZONES.teleport[SAMPLES], 2)} sim units vs a population mean of "
      f"{ZONES.teleport.mean():.2f})")

for letter, m in zip("abcde", MODELS):
    fig = waterfall_grid(
        {a: RES[m]["rolls"][a] for a in ARMS},
        context=obs[:, ef - N_CTX : ef],
        gt=gt_roll,
        title=f"Fig 3{letter} — {m}: each mechanism's own free-run, 4 random episodes (seed 0)",
        sample_idx=SAMPLES,
        target_x=TGT_X,
        ghost_x=GHOST_X,
        metrics={a: RES[m]["cards"][a]["edit_index"] for a in ARMS},
        leads_by_one=(FO,),
    )
    fig.savefig(f"{FIGDIR}/fig3{letter}_waterfall_{RES[m]['spec'].key}.png", dpi=110, bbox_inches="tight")


---
## §2 — Do the two oracle mechanisms move the latent in the same direction?

The central measurement of Part 1. The two mechanisms share nothing but their goal: one rewrites the entire
history and starts the model from scratch, the other appends eight frames to the real history. If their
displacements agree, "the edit direction" is a well-defined object in that latent space rather than an artifact of
how the evidence was delivered.

Two panels' worth of question, one panel: the **whole per-episode distribution** against its own shuffled-pair
control, with the mean drawn on it. A mean-and-error-bar panel would carry strictly less and is not shown.
The table carries the projection fraction and the enrichment — the number to compare **across** architectures,
since `H` runs 64 … 320 and the chance level for `mean |cos|` moves with it.


In [ ]:
# [8] Table 3 + Fig 4 — cos(Counterfactual Overwriting, Freeze-time Interp. TF @8) at the edit frame.
ALIGN_EF = {m: ed.cosine_report(RES[m]["dh_ef"][CF], RES[m]["dh_ef"][FT], seed=0) for m in MODELS}
# The same pair one frame later, so the alignment convention is shown not to drive the answer.
ALIGN_NEXT = {m: ed.cosine_report(RES[m]["dh_next"][CF], RES[m]["dh_next"][FT], seed=0) for m in MODELS}

rows = ["| model | H | cosine (mean ± sd) | angle | projection fraction \\|cos\\| | shuffled \\|cos\\| | "
        "enrichment | cosine at ef+1 |", "|---|---|---|---|---|---|---|---|"]
for m in MODELS:
    r, n = ALIGN_EF[m], ALIGN_NEXT[m]
    rows.append(
        f"| {m} | {r['H']} | **{r['cos_mean']:+.3f}** ± {r['cos_sd']:.3f} | {r['angle_deg']:.0f}° | "
        f"{r['proj_frac']:.3f} | {r['proj_frac_shuffled']:.3f} | **{r['enrichment']:.1f}×** | "
        f"{n['cos_mean']:+.3f} |"
    )
display(Markdown(
    "**Table 3 — the two oracle mechanisms' Δh, compared.** Per episode then averaged, N=256. The shuffled-pair "
    "control pairs each episode's Δh with a *different* episode's, keeping the real distribution of "
    "displacements. The last column repeats the measurement one frame later (every arm free-run one step) — the "
    "alignment convention does not drive the result.\n\n" + "\n".join(rows)
))
print("published reference for the same pair (delta_h_analysis, 2026-08-03, N=256, raw Δh):")
print("  GRU +0.799 · RSSM +0.569 — this notebook's edit-only Δh at the edit frame is the comparable quantity")

fig = F.plot_cosine_violins(
    ALIGN_EF, MODELS,
    title="Fig 4 — do the two oracle mechanisms move the latent in the same direction?",
    pair_label="Counterfactual Overwriting vs Freeze-time Interp. TF @8, at the edit frame",
)
fig.savefig(f"{FIGDIR}/fig4_alignment.png", dpi=140, bbox_inches="tight")


In [ ]:
# [9] Fig 5 — all three mechanisms against each other, one frame after the edit (the common alignment).
P1_ARMS = [CF, FT, FO]
MATS = {m: ed.as_matrix(ed.alignment_matrix(RES[m]["dh_next"], seed=0), P1_ARMS) for m in MODELS}
CHANCE = {m: ALIGN_NEXT[m]["cos_shuffled"] for m in MODELS}

fig = F.plot_cos_matrix(
    MATS, P1_ARMS, MODELS,
    title="Fig 5 — mechanism × mechanism latent alignment, one frame after the edit",
    chance=CHANCE,
)
fig.savefig(f"{FIGDIR}/fig5_matrix.png", dpi=140, bbox_inches="tight")

rows = ["| model | counterfactual vs freeze-time | counterfactual vs first-obs | freeze-time vs first-obs |",
        "|---|---|---|---|"]
for m in MODELS:
    M = MATS[m]
    rows.append(f"| {m} | {M[0, 1]:+.3f} | {M[0, 2]:+.3f} | {M[1, 2]:+.3f} |")
display(Markdown(
    "**Table 4 — every mechanism pair, at the common `ef+1` alignment.** Mean cosine per episode, N=256; the "
    "shuffled-pair chance level for each model is in Fig 5's panel titles.\n\n" + "\n".join(rows)
))


---
## §3 — How big is a successful edit?

A latent distance means nothing on its own, so ‖Δh‖ is reported against two reference scales. **‖h‖** is
meaningful *within* an architecture — is the edit the size of the whole state? **One ordinary dynamics step** is
the scale that transfers *across* architectures — is the edit something the dynamics does anyway, or an excursion
several times larger than any step it takes on its own?


In [ ]:
# [10] Table 5 + Fig 6 — Δh magnitude against the two reference scales.
MAG = {a: [ed.magnitude_report(RES[m]["dh_next"][a], RES[m]["h0_norm"], RES[m]["step_norm"]) for m in MODELS]
       for a in P1_ARMS}

rows = ["| model | mechanism | ‖Δh‖ ÷ ‖h‖ | ‖Δh‖ ÷ one dynamics step | coefficient of variation of ‖Δh‖ |",
        "|---|---|---|---|---|"]
for j, m in enumerate(MODELS):
    for a in P1_ARMS:
        v = MAG[a][j]
        rows.append(f"| {m} | {a} | {v['rel_state_mean']:.2f} ± {v['rel_state_sd']:.2f} | "
                    f"**{v['rel_step_mean']:.1f}** ± {v['rel_step_sd']:.1f} | {v['cv']:.2f} |")
display(Markdown(
    "**Table 5 — the size of a successful edit.** N=256, per episode then averaged. One dynamics step is "
    "‖h_t − h_{t−1}‖ over the last five pre-edit transitions of the same episode.\n\n" + "\n".join(rows)
))

fig = F.plot_magnitudes(
    {a: [v["rel_state_mean"] for v in MAG[a]] for a in P1_ARMS},
    {a: [v["rel_state_sd"] for v in MAG[a]] for a in P1_ARMS},
    {a: [v["rel_step_mean"] for v in MAG[a]] for a in P1_ARMS},
    {a: [v["rel_step_sd"] for v in MAG[a]] for a in P1_ARMS},
    MODELS, P1_ARMS,
    title="Fig 6 — the size of a successful edit, in two reference scales",
)
fig.savefig(f"{FIGDIR}/fig6_magnitude.png", dpi=140, bbox_inches="tight")


---
## §4 — Does the agreement imply structure? Two tests that could break it

Agreement between two mechanisms on the *same* episode is one claim. Two sharper questions decide what it means
for the latent space:

1. **Is there one shared "an object moved" axis?** If the Δh of *different* edits also aligned, there would be a
   generic edit direction and an edit map would be a small object to learn. Chance is 0 here — `1/√H` is the
   per-pair standard deviation, not a floor.
2. **Can a linear position probe see the direction?** The row space of the standard position probe is the entire
   subspace an injection-style write can address, so `‖P_row·Δh‖/‖Δh‖` is a hard ceiling on the cosine any such
   write could reach with the true displacement. Reported ÷ chance `√(d/H)`, because `H` differs by model and the
   raw fraction would otherwise track the moving chance level rather than the geometry.


In [ ]:
# [11] Table 6 + Figs 7 & 8 — a shared edit axis across episodes, and probe visibility of the direction.
CONS = {a: [ed.consistency_report(RES[m]["dh_next"][a], seed=0) for m in MODELS] for a in P1_ARMS}
ROWS = {a: [ed.rowspace_report(RES[m]["dh_next"][a], RES[m]["probe"]["A"]) for m in MODELS] for a in P1_ARMS}

rows = ["| model | mechanism | cross-episode cosine (chance 0) | probe row-space fraction | chance √(d/H) | "
        "enrichment |", "|---|---|---|---|---|---|"]
for j, m in enumerate(MODELS):
    for a in P1_ARMS:
        c, r = CONS[a][j], ROWS[a][j]
        rows.append(f"| {m} | {a} | {c['pairwise_cos_mean']:+.3f} ± {c['pairwise_cos_sd']:.3f} | "
                    f"{r['f_mean']:.3f} | {r['chance']:.3f} | **{r['enrichment']:.2f}×** |")
display(Markdown(
    "**Table 6 — is the direction shared, and is it visible to a probe?** Cross-episode cosine is the mean over "
    "20,000 random pairs of *different* episodes' Δh (± the per-pair spread, so the mean is read against 0). The "
    "probe is the standard linear position read-out, d=4.\n\n" + "\n".join(rows)
))

fig = F.plot_consistency(
    {a: [c["pairwise_cos_mean"] for c in CONS[a]] for a in P1_ARMS},
    {a: [c["pairwise_cos_sd"] for c in CONS[a]] for a in P1_ARMS},
    MODELS, P1_ARMS,
    title="Fig 7 — is there one shared 'an object moved' direction across different edits?",
)
fig.savefig(f"{FIGDIR}/fig7_consistency.png", dpi=140, bbox_inches="tight")

fig = F.plot_rowspace(
    {a: [r["enrichment"] for r in ROWS[a]] for a in P1_ARMS},
    MODELS, P1_ARMS,
    title="Fig 8 — how much of a successful edit can a linear position probe see?",
)
fig.savefig(f"{FIGDIR}/fig8_rowspace.png", dpi=140, bbox_inches="tight")


---
# Part 2 — the two *learned* edit pathways, and whether they write the same displacement

Mechanisms 1 and 2 are oracles: they need a renderer and a rewritten history, and no training distribution makes
them possible or impossible. Mechanisms 3 and 4 are different in kind — they are pathways the model **learned**,
and they exist only for a model whose training data contained teleports:

- **Action Interface** needs an action channel whose action space *contains* the intervention.
- **First Obs. TF** exists for any model, but whether a single uncued post-edit frame *persists* is a fact about
  what the model was trained to expect.

## Which checkpoints support this, and which do not

Every teleport-trained world model in this repo is a **GRU**. Checked against every checkpoint under `runs/`:

| architecture | teleport actions in the action space | teleports present in training data | Part 2 possible |
|---|---|---|---|
| **GRU** | `XG_A_*` (`ActionGRUContinuousModel`, teleport-to-absolute-coordinate actions) | `XG_A_*`, `XG_C_*`, `M_teleport*`, `8_action_cond*` | **yes** |
| RSSM | none | none | no — the only action-conditioned RSSMs (`runs/endogenous_rssm/R*`) take **forces**, not teleports, so their action space cannot express the intervention under test |
| Transformer | none | none | no |
| DiT / latent DiT | none | none | no |

So Part 2 is GRU-only, and that is a fact about the checkpoint inventory rather than about the architectures.
Training the missing models is out of scope here (no models were trained for this notebook); it is what a
follow-up would need to make mechanisms 3 and 4 architecture-independent the way 1 and 2 now are.

## The three models, chosen to isolate what training bought

| label | run code | action channel | teleports in training | what it isolates |
|---|---|---|---|---|
| GRU · teleport actions given | `XG_A_H256` | **yes** | yes (always cued by an action) | all four mechanisms on one model |
| GRU · teleports observed, no action channel | `XG_C_H256` | no | yes (uncued, from the model's point of view) | teleport experience *without* an action channel |
| GRU · never saw a teleport | `H256` | no | no | the same architecture and capacity with neither |

`XG_A` and `XG_C` are the identical recipe on the identical data with the action input removed
(`../action_hidden_size/ACTION_SWEEP_RUNS.md`), so the pair isolates action-knowledge from capacity. The control
is the Part 1 GRU, evaluated on the same episodes.

**Evaluation set.** `datasets/15_teleport_eval_single/eval.h5` — generated with `--p-action 0.0`, so the world
performs **no teleports of its own**; the single edit under test is synthesised at the edit frame and both
ground-truth worlds are rolled forward under passive dynamics. `xg_data` asserts this rather than trusting it
(`GOTCHAS.md`, 2026-08-14).

## The prediction worth stating before the numbers

Sevan's, at the outset: the action-conditioned model may not make an **uncued** teleport persist, because in its
training every teleport arrived with an action. That is a prediction about `First Obs. TF` specifically, and the
`XG_A` / `XG_C` pair is exactly the comparison that tests it.


In [ ]:
# [12] Part 2 setup — the teleport-free eval episodes with one synthesised teleport, and the three GRUs.
from eval_action_sweep import EF as XG_EF
from eval_action_sweep import xg_data

SPECS2 = [
    ed.ModelSpec("xga", "GRU · teleport actions given (H=256)", "runs/action_sweep/XG_A_H256/best_model.pt",
                 "GRU", "hidden state h", run_code="XG_A_H256", loader="action_sweep",
                 registry="../action_hidden_size/ACTION_SWEEP_RUNS.md"),
    ed.ModelSpec("xgc", "GRU · teleports observed, no action channel (H=256)",
                 "runs/action_sweep/XG_C_H256/best_model.pt", "GRU", "hidden state h",
                 run_code="XG_C_H256", loader="action_sweep",
                 registry="../action_hidden_size/ACTION_SWEEP_RUNS.md"),
    ed.ModelSpec("ctrl", "GRU · never saw a teleport (H=256)", "runs/controls/H256/best_model.pt",
                 "GRU", "hidden state h", run_code="H256", registry="../controls/CONTROL_RUNS.md"),
]
MODELS2 = [s.label for s in SPECS2]
ARMS2 = [UNS, CF, FT, AC, FO]

E = xg_data(n_edits=N_EVAL, n_probe=N_PROBE, seed=0)
sim2 = E["sim"].__dict__
EV2 = ed.build_evidence(
    positions=E["pos"], velocities=E["vel"], edit_object=E["edit_obj"], target=E["tgt"],
    sim=sim2, ef=XG_EF, n_obj=N_OBJ, n_ft=N_FT,
    uned_pos=E["pos"][:, XG_EF],  # exact: this world performs no teleports of its own
    seed=0,
)
ZONES2, GT2 = E["zones"], E["gt_roll"]
ZONES2_1 = shift_zones(ZONES2, 1, GT2)
probe_pos2 = E["probe_pos"].reshape(N_PROBE, -1, N_OBJ * 2)
print(f"edit frame {XG_EF} · teleport {ZONES2.teleport.mean():.2f} sim units · "
      f"differing {ZONES2.differing.sum(1).mean():.1f} of {R} rays")


In [ ]:
# [13] One pass per model: up to four mechanisms, their rollouts, scorecards, probe, and deltas.
RES2 = {}
for spec in SPECS2:
    t0 = time.perf_counter()
    model = ed.load_model(spec, root=".", device=DEVICE)
    has_actions = hasattr(model, "action_proj")  # only the action-conditioned GRU has the channel

    run = ed.build_states(
        model, spec, EV2, E["obs"], XG_EF, device=DEVICE,
        actions_edit=E["act_edit"] if has_actions else None,
        actions_noop=E["act_noop"] if has_actions else None,
    )
    arms = [UNS] + [a for a in (CF, FT, AC, FO) if a in run.mechanisms]
    rolls = {UNS: ed.rollout(model, run.unsteered, K_ROLL)}
    for arm in arms[1:]:
        rolls[arm] = ed.rollout(model, run.mechanisms[arm].edited, K_ROLL)

    uns_lead = edit_scorecard(
        ed.rollout(model, ed.advance(model, run.unsteered, 1), K_ROLL - 1), ZONES2_1, GT2[:, 1:]
    )
    cards = {}
    for arm, roll in rolls.items():
        lead = arm == FO
        z, g = (ZONES2_1, GT2[:, 1:]) if lead else (ZONES2, GT2)
        cards[arm] = edit_scorecard(roll[:, : g.shape[1]], z, g)
        cards[arm]["fidelity_ratio"] = fidelity_ratio(cards[arm], uns_lead if lead else cards[UNS])

    RES2[spec.label] = dict(
        spec=spec, arms=arms, cards=cards, rolls=rolls,
        probe=ed.fit_position_probe(model, E["probe_obs"], probe_pos2, device=DEVICE, seed=0),
        dh_ef=ed.deltas(model, run, align_to_edit_frame=True),
        dh_next=ed.deltas(model, run, align_to_edit_frame=False),
        h0_norm=run.h0_norm, step_norm=run.step_norm,
    )
    print(f"{spec.label:52s} action channel {str(has_actions):5s}  {time.perf_counter() - t0:4.0f}s")
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


---
## §5 — Do all four mechanisms work on one model?

The first question Part 2 has to answer before any geometry: is there a single checkpoint on which all four
mechanisms edit the generation? If `Action Interface` or `First Obs. TF` fails to land, its Δh is not a successful
edit's displacement and comparing it to the oracles' would be measuring nothing.


In [ ]:
# [14] Table 7 + Figs 9 & 10 — the gate for all four mechanisms on the three teleport-trained GRUs.
rows = ["| model | mechanism | Edit Index (step 0) ↑ | Edit Index (step 14) ↑ | GT-traj RMSE ↓ | fidelity ↓ |",
        "|---|---|---|---|---|---|"]
for m in MODELS2:
    for arm in ARMS2:
        if arm not in RES2[m]["cards"]:
            rows.append(f"| {m} | {arm} | n/a — no action channel | n/a | n/a | n/a |")
            continue
        c = RES2[m]["cards"][arm]
        rows.append(f"| {m} | {arm} | **{c['edit_index']:+.3f}** | {c['edit_index_by_step'][-1]:+.3f} | "
                    f"{c['gt_traj_rmse']:.3f} | {c['fidelity_ratio']:.2f} |")
display(Markdown(
    "**Table 7 — the four mechanisms on the teleport-trained models.** N=256 episodes from a teleport-free eval "
    "world with one synthesised edit. `Action Interface` reads `n/a` for the two models without an action "
    "channel — a structural absence, not a missing measurement.\n\n" + "\n".join(rows)
))

def by_arm(field):
    return {a: [RES2[m]["cards"].get(a, {}).get(field, np.nan) for m in MODELS2] for a in ARMS2}

fig = F.plot_edit_gate(
    by_arm("edit_index"), by_arm("fidelity_ratio"), MODELS2, ARMS2,
    title=f"Fig 9 — all four mechanisms, on models that differ only in what teleports they saw  (N={N_EVAL})",
)
fig.savefig(f"{FIGDIR}/fig9_gate_part2.png", dpi=140, bbox_inches="tight")

fig = F.plot_index_by_step(
    {m: {a: RES2[m]["cards"][a]["edit_index_by_step"] for a in RES2[m]["arms"]} for m in MODELS2},
    MODELS2, ARMS2,
    title="Fig 10 — Edit Index across the rollout: does an uncued teleport persist?",
)
fig.savefig(f"{FIGDIR}/fig10_by_step_part2.png", dpi=140, bbox_inches="tight")


In [ ]:
# [15] Fig 11a-c — the waterfall for each Part 2 model, same random episodes (seed 0) in all three.
SAMPLES2 = random_samples(N_EVAL, 4, seed=0)
TGT_X2, GHOST_X2 = ray_centre(ZONES2.target), ray_centre(ZONES2.ghost)
print(f"random episodes shown: {SAMPLES2}  (teleports {np.round(ZONES2.teleport[SAMPLES2], 2)} sim units "
      f"vs a population mean of {ZONES2.teleport.mean():.2f})")

for letter, m in zip("abc", MODELS2):
    fig = waterfall_grid(
        {a: RES2[m]["rolls"][a] for a in RES2[m]["arms"]},
        context=E["obs"][:, XG_EF - N_CTX : XG_EF],
        gt=GT2,
        title=f"Fig 11{letter} — {m}: each mechanism's own free-run, 4 random episodes (seed 0)",
        sample_idx=SAMPLES2,
        target_x=TGT_X2,
        ghost_x=GHOST_X2,
        metrics={a: RES2[m]["cards"][a]["edit_index"] for a in RES2[m]["arms"]},
        leads_by_one=(FO,),
    )
    fig.savefig(f"{FIGDIR}/fig11{letter}_waterfall_{RES2[m]['spec'].key}.png", dpi=110, bbox_inches="tight")


---
## §6 — Do the learned pathways write the same displacement as the oracles?

The question Part 2 exists for. The action channel was **trained**; the oracles were not. If the action-induced
displacement lands on the same direction as a rewritten history, then the learned pathway and the oracle are two
routes to one latent edit rather than two different edits that happen to produce similar pictures.

All four arms are compared at the common `ef+1` alignment (mechanisms 1–3 free-run one step; `First Obs. TF` is
already there), and every pair is read against that model's own shuffled-pair control.


In [ ]:
# [16] Table 8 + Fig 12 — every mechanism pair, on every Part 2 model.
P2_ARMS = [CF, FT, AC, FO]
MATS2, CHANCE2, PAIRS2 = {}, {}, {}
for m in MODELS2:
    mat = ed.alignment_matrix(RES2[m]["dh_next"], seed=0)
    present = [a for a in P2_ARMS if a in RES2[m]["dh_next"]]
    M = np.full((len(P2_ARMS), len(P2_ARMS)), np.nan)
    for i, a in enumerate(P2_ARMS):
        for j, b in enumerate(P2_ARMS):
            if a in present and b in present:
                M[i, j] = 1.0 if a == b else mat[a][b]["cos_mean"]
    MATS2[m], PAIRS2[m] = M, mat
    CHANCE2[m] = mat[present[0]][present[1]]["cos_shuffled"]

fig = F.plot_cos_matrix(
    MATS2, P2_ARMS, MODELS2,
    title="Fig 12 — mechanism × mechanism latent alignment on the teleport-trained models",
    chance=CHANCE2,
)
fig.savefig(f"{FIGDIR}/fig12_matrix_part2.png", dpi=140, bbox_inches="tight")

rows = ["| model | pair | cosine ± sd | angle | projection fraction \\|cos\\| | shuffled \\|cos\\| | enrichment |",
        "|---|---|---|---|---|---|---|"]
for m in MODELS2:
    present = [a for a in P2_ARMS if a in RES2[m]["dh_next"]]
    for i, a in enumerate(present):
        for b in present[i + 1:]:
            r = PAIRS2[m][a][b]
            rows.append(f"| {m} | {a} vs {b} | **{r['cos_mean']:+.3f}** ± {r['cos_sd']:.3f} | "
                        f"{r['angle_deg']:.0f}° | {r['proj_frac']:.3f} | {r['proj_frac_shuffled']:.3f} | "
                        f"**{r['enrichment']:.1f}×** |")
display(Markdown(
    "**Table 8 — every mechanism pair on the teleport-trained models.** Per episode then averaged, N=256, at the "
    "common `ef+1` alignment.\n\n" + "\n".join(rows)
))


In [ ]:
# [17] Table 9 + Fig 13 — Part 2 magnitudes, shared-axis test, and probe visibility.
MAG2 = {a: [ed.magnitude_report(RES2[m]["dh_next"][a], RES2[m]["h0_norm"], RES2[m]["step_norm"])
            if a in RES2[m]["dh_next"] else None for m in MODELS2] for a in P2_ARMS}
ROWS2 = {a: [ed.rowspace_report(RES2[m]["dh_next"][a], RES2[m]["probe"]["A"])
             if a in RES2[m]["dh_next"] else None for m in MODELS2] for a in P2_ARMS}
CONS2 = {a: [ed.consistency_report(RES2[m]["dh_next"][a], seed=0)
             if a in RES2[m]["dh_next"] else None for m in MODELS2] for a in P2_ARMS}

rows = ["| model | mechanism | ‖Δh‖ ÷ ‖h‖ | ‖Δh‖ ÷ one dynamics step | cross-episode cosine | "
        "probe row-space ÷ chance |", "|---|---|---|---|---|---|"]
for j, m in enumerate(MODELS2):
    for a in P2_ARMS:
        if MAG2[a][j] is None:
            rows.append(f"| {m} | {a} | n/a | n/a | n/a | n/a |")
            continue
        v, r, c = MAG2[a][j], ROWS2[a][j], CONS2[a][j]
        rows.append(f"| {m} | {a} | {v['rel_state_mean']:.2f} | **{v['rel_step_mean']:.1f}** | "
                    f"{c['pairwise_cos_mean']:+.3f} ± {c['pairwise_cos_sd']:.3f} | **{r['enrichment']:.2f}×** |")
display(Markdown(
    "**Table 9 — Part 2 geometry.** Same definitions as Tables 5 and 6. `n/a` marks the two models with no "
    "action channel.\n\n" + "\n".join(rows)
))

nan = float("nan")
fig = F.plot_rowspace(
    {a: [r["enrichment"] if r else nan for r in ROWS2[a]] for a in P2_ARMS},
    MODELS2, P2_ARMS,
    title="Fig 13 — probe visibility of each mechanism's direction, including the learned action pathway",
)
fig.savefig(f"{FIGDIR}/fig13_rowspace_part2.png", dpi=140, bbox_inches="tight")


---
# Current results (updated 2026-08-19)

*Everything above this line is the invariant pipeline; everything in this block is what it currently reads.*

## 1 — Every working edit points the same way, in every architecture

`cos(Counterfactual Overwriting, Freeze-time Interp. TF @8)` at the edit frame, N=256, per episode then averaged
(Table 3, Fig 4). The two mechanisms share nothing but their goal — one rewrites the whole history and restarts
the model, the other appends eight frames to the real history:

| state object | cosine | angle | enrichment over shuffled |
|---|---|---|---|
| DiT (pixel) · residual stream | **+0.910** | 25° | 5.5× |
| GRU · hidden state | **+0.808** | 36° | 5.2× |
| Transformer · residual stream | **+0.806** | 36° | 4.4× |
| Latent DiT · latent window | **+0.667** | 48° | 4.0× |
| RSSM · det+stoch state | **+0.593** | 54° | 4.4× |

The GRU and RSSM values replicate `delta_h_analysis` (2026-08-03: +0.799 / +0.569) on an independent
construction of the Δh, and the result now holds on three architectures that notebook never touched. Reading the
angles rather than the cosines keeps it honest: 25°–54° is *strongly* aligned against a shuffled control of
+0.00 ± 0.22, not identical. "The edit direction" is a well-defined object in every latent space tested.

## 2 — The learned action channel writes the *same* displacement as the oracle

On `XG_A_H256`, the one model where all four mechanisms are available, **all four land** (Table 7): Edit Index
+0.643 counterfactual · +0.563 freeze-time · **+0.645 action interface** · +0.216 first-obs, against an unsteered
−0.641, all with fidelity ≤ 0.66. The action interface is the strongest arm and the best at *holding* the edit
(+0.473 at step 14 vs the counterfactual's +0.409).

Their displacements agree (Table 8, Fig 12), and the tightest pair in the entire notebook is
**counterfactual overwrite vs the trained action channel: +0.872 (29°), 5.9× chance.** A pathway the model
*learned* from data and an oracle that rewrites its history arrive at nearly the same latent displacement. Every
pair on this model sits between +0.72 and +0.87.

## 3 — Whether one uncued frame sticks is a fact about the training distribution

`First Obs. TF` — one post-edit observation, no action, no rewritten history — at step 0, on three GRUs that
differ only in what they saw during training:

| model | Edit Index (step 0) | at step 14 |
|---|---|---|
| never saw a teleport | **−0.002** | −0.095 |
| teleport actions given | **+0.216** | +0.162 |
| teleports observed, no action channel | **+0.532** | +0.335 |

Sevan's prediction was right, and in its sharpest form: the **action-conditioned** model does not commit to an
**uncued** teleport (+0.22), while the model trained on the identical teleports with the action input removed
does (+0.53). For `XG_A` every teleport in training arrived with an action, so an unexplained jump is evidence
of noise; for `XG_C` an unexplained jump was the only kind there was. The control that never saw a teleport
does not commit at all. Same architecture, same capacity, same data for the first two — only the cue differs.

## 4 — The agreement does not come with the structure that would make editing easy

Two tests that could have broken the story, and their answers do not soften it (Table 6, Figs 7 & 8):

- **No shared "an object moved" axis.** The mean cosine between the Δh of *different* episodes is +0.00 … +0.04
  in every model and every mechanism, against a chance level of 0. Two mechanisms agree about *the same* edit;
  there is no generic edit direction to learn. This replicates `delta_h_analysis` §5 (+0.011) across four more
  state objects.
- **The direction is invisible to a linear position probe.** Row-space fraction ÷ chance: GRU 0.73×,
  transformer 0.49×, pixel DiT 0.14×, RSSM 0.03× — at or **below** the level a random direction would score.
  The one exception is the **latent DiT's carried latent window at 1.17×** (1.46× for first-obs): in a 64-d
  learned code the edit is, marginally, more probe-visible than chance. It is the only state object in the study
  where that is true, and 1.17× is a long way from a handle.
- **Magnitude** (Table 5): a successful edit is 2.4–6.4 × one ordinary dynamics step. Within the recurrent
  models it is about as large as the whole state (‖Δh‖/‖h‖ 0.94–1.04); on the residual-stream views it is 0.2–0.4
  of the stream's norm, which is why the dynamics-step scale is the one to compare across architectures.

## 5 — What this adds up to

The four mechanisms are not four different edits that happen to produce similar pictures; they are four routes to
**one** displacement, and that displacement is well defined per episode, large, idiosyncratic across episodes, and
almost entirely outside what a linear read-out of position can address. The consistent story across the whole
editability thread — that the barrier is not reachability or capacity but the absence of a *map from the intended
change to the required state change* — now has its positive half measured on four architectures: the map exists
and is well defined; it is simply not a probe direction, and not a single shared axis.

The action-channel result is the constructive one. It is the first demonstration here that a **learned** pathway
lands on the oracle's displacement, which makes "train something that emits Δh" a well-posed target rather than a
hope.

## Caveats, stated with the results

- **The RSSM is the outlier everywhere, and part of that is its weak freeze-time arm.** Freeze-time reaches only
  +0.097 on the RSSM (vs +0.52 … +0.65 elsewhere), so its Δh is partly the displacement of an edit that did not
  land — the most likely reason its cosine is the lowest in Table 3. Its row-space fraction (0.03× chance, a 30×
  *depletion*) also deserves a dedicated look rather than being read as a stronger version of the same effect.
- **One checkpoint per architecture, one seed, N=256, one world.** Nothing here separates architecture from
  checkpoint. Two points make a line, not a law, and five do not make it a theorem.
- **Part 2 is GRU-only** because no teleport-trained RSSM, transformer, or DiT exists in this repo. Mechanisms 3
  and 4 are therefore untested for architecture-independence, unlike 1 and 2.
- **Alignment is measured, not assumed, and two models sit at `k = −1`.** The latent DiT and pixel DiT hedge
  toward the frame they just consumed, so their `k = 0` value is not their minimum; it does reproduce their
  published next-step RMSE against the clean render (0.1080 vs 0.1083, 0.1088 vs 0.1089), which is the check that
  settles alignment. If those two models were nonetheless one frame off, their rows in every table would be
  measuring a slightly different quantity from the rest.
- **Δh is always `edited − matched control`**, never `edited − unsteered`. That is the stronger construction, but
  it means these numbers are not directly comparable to any "raw Δh" reported elsewhere without checking which
  was used.

## What would falsify or sharpen this

1. A teleport-trained transformer or RSSM would make mechanisms 3 and 4 architecture-independent, and would say
   whether "the action channel writes the oracle's displacement" is a GRU fact or a general one.
2. Corrupting the action channel's write while holding its read-out accuracy fixed would test whether the
   agreement in §2 is doing causal work or is a correlate of both mechanisms landing the same picture.
3. The RSSM's near-zero row-space fraction and its weak freeze-time arm are the two loose threads; both are
   cheap to pull.
